In [ ]:
import os
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
from collections import Counter

In [3]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

In [5]:
text = "machine"

In [6]:
embedding = model.encode(text)

In [7]:
len(embedding)

384

In [8]:
embedding.shape

(384,)

In [9]:
sentence = "Machine learning is a subset of artificial intelligence"

embedding = model.encode(sentence)

len(embedding)

384

In [11]:
# embedding

In [12]:
paragraph = """ 
Machine learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers 
to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make decisions with minimal human 
intervention. Machine learning is widely used in various applications, including image recognition, natural language processing, and recommendation systems.
"""

embedding = model.encode(paragraph)

len(embedding)

384

In [13]:
embedding.shape

(384,)

In [15]:
# embedding

## Google Embeddings

In [16]:
google_embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [17]:
google_embedding = google_embeddings_model.embed_query("What is the capital of France?")

In [19]:
len(google_embedding)

3072

## OpenAI Embedding

In [ ]:
openai_embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [21]:
openai_embedding = openai_embeddings_model.embed_query("What is the capital of France?")

In [22]:
len(openai_embedding)

3072

In [23]:
openai_embeddings_small_model = OpenAIEmbeddings(model="text-embedding-3-small")
openai_small_embedding = openai_embeddings_small_model.embed_query("What is the capital of France?")

In [24]:
len(openai_small_embedding)

1536

## CLIP Model

In [25]:
image_path = "data/transformer_architecture.png"

In [26]:
image = Image.open(image_path)

In [27]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [28]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

In [29]:
inputs = processor(images=image, return_tensors="pt")

In [30]:
with torch.no_grad():
    image_embedding = model.get_image_features(**inputs)

image_embedding

tensor([[-3.1277e-02, -2.2570e-01,  8.3339e-02, -2.0438e-01, -4.6600e-01,
         -4.2965e-01,  2.1990e-01,  2.0938e-01,  3.2516e-01, -4.1802e-02,
          1.3474e-01,  1.9919e-01,  2.3379e-01,  4.2718e-01, -2.6818e-01,
         -1.2241e-01,  3.9351e-01,  4.0981e-01, -4.8721e-01,  6.4884e-02,
          1.6097e-01,  1.4812e-01,  4.1753e-01,  1.9362e-01, -8.2574e-01,
          2.1905e-01, -2.9516e-02,  1.9250e-01,  1.0967e-01, -2.7309e-03,
         -1.5061e-01,  9.7995e-02,  7.9381e-02,  3.2859e-01,  5.4545e-03,
         -8.4180e-02, -4.5888e-02,  4.6035e-01, -2.5762e-01, -2.0197e+00,
         -6.4738e-01,  3.0611e-01, -2.0147e-01, -7.6115e-01,  5.9522e-01,
         -1.8778e-01,  4.0526e-03,  8.7673e-02,  5.1040e-02,  1.4128e-01,
          5.5078e-01, -1.3085e-01,  3.7968e-01,  2.7426e-01,  1.3154e-02,
          2.3552e-01,  2.0026e-03,  5.0496e-02,  4.5904e-01,  2.3715e-01,
          5.8229e-02, -1.2206e-01,  5.4171e-01,  5.9222e-03, -1.3182e-01,
          6.7514e-02, -1.5713e-01,  5.

In [31]:
image_embedding.shape

torch.Size([1, 512])

In [32]:
len(image_embedding[0])

512

## Similarity/Semantic Search

1. Dot Product
2. Cosine Sim
3. Eculidean Distance(L2)

In [33]:
def dot_product(a, b):
    return np.dot(a, b)

In [34]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [35]:
def euclidean_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.linalg.norm(a - b)

In [36]:
print("=== Cosine Similarity (Higher is better) ===")
print("Range: -1 to 1")

print("\n=== Dot Product (Higher is better) ===")
print("Range: -∞ to +∞ (depends on vector magnitude)")

print("\n=== Euclidean Distance (Lower is better) ===")
print("Range: 0 to +∞")

=== Cosine Similarity (Higher is better) ===
Range: -1 to 1

=== Dot Product (Higher is better) ===
Range: -∞ to +∞ (depends on vector magnitude)

=== Euclidean Distance (Lower is better) ===
Range: 0 to +∞


In [37]:
query = "How to reduce heart disease risk?"

In [38]:
documents = [
    "Eating fiber reduces heart risk.",
    "Fruits and vegetables lower cardiovascular disease chances.",
    "Buying a new car improves driving comfort.",
    "Regular exercise improves heart health."
]

In [39]:
query_embedding = openai_embeddings_model.embed_query(query)

In [40]:
documents_embeddings = openai_embeddings_model.embed_documents(documents)

In [41]:
len(documents_embeddings)

4

In [42]:
len(documents_embeddings[0])

3072

In [49]:
results = []

for document, documents_embedding in zip(documents, documents_embeddings):
    results.append({
        "query": query,
        "document": document,
        "cosine": cosine_similarity(query_embedding, documents_embedding),
        "dot_product": dot_product(query_embedding, documents_embedding),
        "euclidean": euclidean_distance(query_embedding, documents_embedding)
    })

In [50]:
results

[{'query': 'How to reduce heart disease risk?',
  'document': 'Eating fiber reduces heart risk.',
  'cosine': 0.5985719577977843,
  'dot_product': 0.5986622087840239,
  'euclidean': 0.8960899733874286},
 {'query': 'How to reduce heart disease risk?',
  'document': 'Fruits and vegetables lower cardiovascular disease chances.',
  'cosine': 0.5579075891767901,
  'dot_product': 0.5578281944618197,
  'euclidean': 0.9402441150573565},
 {'query': 'How to reduce heart disease risk?',
  'document': 'Buying a new car improves driving comfort.',
  'cosine': 0.06057430919232074,
  'dot_product': 0.060568684150503316,
  'euclidean': 1.3706483553936326},
 {'query': 'How to reduce heart disease risk?',
  'document': 'Regular exercise improves heart health.',
  'cosine': 0.4914256906935539,
  'dot_product': 0.491606805490445,
  'euclidean': 1.0087238371978915}]

## Keyword Search

In [51]:
documents

['Eating fiber reduces heart risk.',
 'Fruits and vegetables lower cardiovascular disease chances.',
 'Buying a new car improves driving comfort.',
 'Regular exercise improves heart health.']

In [52]:
query

'How to reduce heart disease risk?'

In [54]:
query.lower().replace("?", "").split()

['how', 'to', 'reduce', 'heart', 'disease', 'risk']

In [55]:
query_words = query.lower().replace("?", "").split()

In [56]:
documents[0]

'Eating fiber reduces heart risk.'

In [57]:
document_words = documents[0].lower().replace(".", "").split()
document_words

['eating', 'fiber', 'reduces', 'heart', 'risk']

In [58]:
document_count = Counter(document_words)
document_count

Counter({'eating': 1, 'fiber': 1, 'reduces': 1, 'heart': 1, 'risk': 1})

In [59]:
for word in query_words:
    print(f"{word}: {document_count[word]}")

how: 0
to: 0
reduce: 0
heart: 1
disease: 0
risk: 1


### Score = number of matching words

In [61]:
score = sum(document_count[word] for word in query_words)
score

2

In [62]:
def keyword_search(query, documents):
    query_words = query.lower().replace("?", "").split()
    results = []

    for document in documents:
        document_words = document.lower().replace(".", "").split()
        document_count = Counter(document_words)

        score = sum(document_count[word] for word in query_words) 
        results.append((document, score))

    return sorted(results, key=lambda x: x[1], reverse=True)

In [63]:
keyword_results = keyword_search(query, documents)

keyword_results

[('Eating fiber reduces heart risk.', 2),
 ('Fruits and vegetables lower cardiovascular disease chances.', 1),
 ('Regular exercise improves heart health.', 1),
 ('Buying a new car improves driving comfort.', 0)]